In [1]:
!pip install torchcodec
!pip install librosa torchaudio
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.3 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset , Dataset
import torch
import torchaudio.transforms as T

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [4]:
dataset = load_dataset("Purvaxxx/ASR")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/hindi-00000-of-00006.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

data/hindi-00001-of-00006.parquet:   0%|          | 0.00/515M [00:00<?, ?B/s]

data/marathi-00000-of-00003.parquet:   0%|          | 0.00/362M [00:00<?, ?B/s]

...

Generating hindi split:   0%|          | 0/88566 [00:00<?, ? examples/s]

Generating marathi split:   0%|          | 0/29154 [00:00<?, ? examples/s]

Generating telugu split:   0%|          | 0/58344 [00:00<?, ? examples/s]

Generating bengali split:   0%|          | 0/86763 [00:00<?, ? examples/s]

DatasetDict({
    hindi: Dataset({
        features: ['chunked_audio_filepath', 'text', 'pred_text', 'audio_filepath', 'start_time', 'duration', 'alignment_score', 'en_text', 'en_mining_score'],
        num_rows: 88566
    })
    marathi: Dataset({
        features: ['chunked_audio_filepath', 'text', 'pred_text', 'audio_filepath', 'start_time', 'duration', 'alignment_score', 'en_text', 'en_mining_score'],
        num_rows: 29154
    })
    telugu: Dataset({
        features: ['chunked_audio_filepath', 'text', 'pred_text', 'audio_filepath', 'start_time', 'duration', 'alignment_score', 'en_text', 'en_mining_score'],
        num_rows: 58344
    })
    bengali: Dataset({
        features: ['chunked_audio_filepath', 'text', 'pred_text', 'audio_filepath', 'start_time', 'duration', 'alignment_score', 'en_text', 'en_mining_score'],
        num_rows: 86763
    })
})


In [5]:
# for now working with dataset and removing the unwanted columns
hindi_dataset = dataset["hindi"].remove_columns ([
    "audio_filepath",
    "start_time",
    "alignment_score",
    "duration",
    "en_text",
    "en_mining_score"
    ])

In [6]:
print(hindi_dataset) # keeping only audio and the source text

Dataset({
    features: ['chunked_audio_filepath', 'text', 'pred_text'],
    num_rows: 88566
})


In [7]:
# Taking the first 10k subset
subset = hindi_dataset.select(range(10000))

# Collect all transcriptions
all_text = "".join(subset["text"]).lower()

# Unique sorted characters
characters = sorted(list(set(all_text)))

# Add blank token for CTC
characters = ["<Blank>"] + characters

# Create mapping dictionaries
char_to_num = {char: idx for idx, char in enumerate(characters)}
num_to_char = {idx: char for idx, char in enumerate(characters)}

print("Vocab:", characters)
print("Vocab size:", len(characters))


Vocab: ['<Blank>', ' ', '"', "'", 'ँ', 'ं', 'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ऋ', 'ए', 'ऐ', 'ओ', 'औ', 'क', 'ख', 'ग', 'घ', 'च', 'छ', 'ज', 'झ', 'ञ', 'ट', 'ठ', 'ड', 'ढ', 'ण', 'त', 'थ', 'द', 'ध', 'न', 'प', 'फ', 'ब', 'भ', 'म', 'य', 'र', 'ल', 'व', 'श', 'ष', 'स', 'ह', '़', 'ा', 'ि', 'ी', 'ु', 'ू', 'ृ', 'ॅ', 'े', 'ै', 'ॉ', 'ो', 'ौ', '्', 'ॠ']
Vocab size: 64


In [8]:
spectrogram_transform = T.Spectrogram(
    n_fft=256,
    hop_length=160,
    win_length=256,
    power=2.0
).to(device)

def preprocess_fn(batch):
    audio_array = batch["chunked_audio_filepath"]["array"]
    transcription = batch["text"]
    predicted_text = batch.get("pred_text", "")
    # Torch tensor on GPU
    waveform = torch.tensor(audio_array, dtype=torch.float32, device=device)

    # Spectrogram
    spectrogram = spectrogram_transform(waveform)
    spectrogram = torch.sqrt(spectrogram).permute(1, 0)

    # Normalize
    means = spectrogram.mean(dim=0, keepdim=True)
    stddevs = spectrogram.std(dim=0, keepdim=True)
    spectrogram = (spectrogram - means) / (stddevs + 1e-10)

    # Encode label
    label = [char_to_num[char] for char in transcription.lower()]

    # Return as CPU numpy/tensor (datasets stores on CPU)
    return {
        "spectrogram": spectrogram.cpu().numpy(),
        "label": label,
        "pred_text": predicted_text
    }


In [9]:
processed_dataset = subset.map(
    preprocess_fn,
    remove_columns=subset.column_names,
    desc="Preprocessing audio"
)

Preprocessing audio:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [12]:
print(processed_dataset)

Dataset({
    features: ['pred_text', 'spectrogram', 'label'],
    num_rows: 10000
})


In [ ]:
processed_dataset.push_to_hub("Purvaxxx/hindi_CTC")